In [ ]:
import sqlite3
import pandas as pd

# Abrir a conexão com o banco
conn = sqlite3.connect("lh_nautical.db")
print("Conexão aberta!")

Conexão aberta!


In [48]:
query = """
SELECT
    full_name AS cliente,
    cidade,
    estado,
    COUNT(*) AS total_compras,
    SUM(qtd) AS total_itens,
    ROUND(SUM(total), 2) AS total_faturado,
    ROUND(AVG(total), 2) AS ticket_medio

FROM base_completa
GROUP BY id_client, full_name, cidade, estado
ORDER BY total_faturado DESC
LIMIT 10
"""

df_top_clientes = pd.read_sql(query, conn)
df_top_clientes["total_faturado"] = df_top_clientes["total_faturado"].apply(lambda x: f"R$ {x:,.2f}")
df_top_clientes["ticket_medio"] = df_top_clientes["ticket_medio"].apply(lambda x: f"R$ {x:,.2f}")
print("TOP 10 CLIENTES")
print(df_top_clientes.to_string(index=False))

TOP 10 CLIENTES
                               cliente               cidade estado  total_compras  total_itens   total_faturado  ticket_medio
                     Márcia Figueiredo        Vila Do Conde     PA            222         1851 R$ 72,187,369.50 R$ 325,168.33
              Lucas Guedes Cunha Lopes          João Pessoa     PB            218         1691 R$ 66,788,855.35 R$ 306,370.90
  Fernanda Azevedo Soares Nunes Vieira               Recife     PE            220         2024 R$ 65,652,931.35 R$ 298,422.42
       Carla Lopes Alves Pacheco Rocha Fortaleza Do Tabocão     TO            233         1934 R$ 64,171,776.55 R$ 275,415.35
               Gabriela Barros Lacerda              Niterói     RJ            190         1544 R$ 64,003,343.75 R$ 336,859.70
            Francisca Ribeiro Pinheiro                Belém     PA            215         1752 R$ 62,791,038.15 R$ 292,051.34
Femininos Antunes Lopes Ribeiro Amaral           São Mateus     ES            217         1947 R$ 62,0

In [47]:
query = """
SELECT
    full_name AS cliente,
    COUNT(*) AS frequencia,
    ROUND(SUM(total), 2) AS valor_total,
    MAX(sale_date) AS ultima_compra,
    CASE
        WHEN SUM(total) >= 55000000 THEN 'Alto Valor'
        WHEN SUM(total) >= 50000000  THEN 'Médio Valor'
        ELSE 'Baixo Valor'
    END AS segmento

FROM base_completa
GROUP BY id_client, full_name
ORDER BY valor_total DESC
"""

df_segmentos = pd.read_sql(query, conn)
df_segmentos["valor_total"] = df_segmentos["valor_total"].apply(lambda x: f"R$ {x:,.2f}")
print("SEGMENTAÇÃO DE CLIENTES")
print(df_segmentos.to_string(index=False))

# Resumo por segmento
print("\nResumo por segmento:")
print(df_segmentos["segmento"].value_counts().to_string())

SEGMENTAÇÃO DE CLIENTES
                                    cliente  frequencia      valor_total ultima_compra    segmento
                          Márcia Figueiredo         222 R$ 72,187,369.50    2024-12-29  Alto Valor
                   Lucas Guedes Cunha Lopes         218 R$ 66,788,855.35    2024-12-30  Alto Valor
       Fernanda Azevedo Soares Nunes Vieira         220 R$ 65,652,931.35    2024-12-31  Alto Valor
            Carla Lopes Alves Pacheco Rocha         233 R$ 64,171,776.55    2024-12-29  Alto Valor
                    Gabriela Barros Lacerda         190 R$ 64,003,343.75    2024-12-28  Alto Valor
                 Francisca Ribeiro Pinheiro         215 R$ 62,791,038.15    2024-12-31  Alto Valor
     Femininos Antunes Lopes Ribeiro Amaral         217 R$ 62,028,628.95    2024-12-30  Alto Valor
                           Bianca Rodrigues         204 R$ 60,826,837.25    2024-12-26  Alto Valor
      Daniela Borges Vieira Farias Mendonça         198 R$ 59,581,398.75    2024-12-2

In [45]:
query = """
WITH segmentacao AS (
    SELECT
        id_client,
        full_name,
        COUNT(*) AS frequencia,
        ROUND(SUM(total), 2) AS valor_total,
        CASE
            WHEN SUM(total) >= 55000000 THEN 'Alto Valor'
            WHEN SUM(total) >= 50000000     THEN 'Médio Valor'
            ELSE 'Baixo Valor'
        END AS segmento
    FROM base_completa
    GROUP BY id_client, full_name
)

SELECT
    segmento,
    COUNT(*) AS total_clientes,
    ROUND(AVG(frequencia), 2) AS media_compras,
    ROUND(
        SUM(frequencia) * 100.0 / SUM(SUM(frequencia)) OVER (),
    2) AS pct_total_compras,
    ROUND(AVG(valor_total), 2) AS ticket_medio_segmento

FROM segmentacao
GROUP BY segmento
ORDER BY ticket_medio_segmento DESC
"""

df_freq = pd.read_sql(query, conn)

df_freq["ticket_medio_segmento"] = df_freq["ticket_medio_segmento"].apply(lambda x: f"R$ {x:,.2f}")
df_freq["pct_total_compras"] = df_freq["pct_total_compras"].apply(lambda x: f"{x}%")

print("FREQUENCIA DE COMPRAS POR SEGMENTO")
print(df_freq.to_string(index=False))

FREQUENCIA DE COMPRAS POR SEGMENTO
   segmento  total_clientes  media_compras pct_total_compras ticket_medio_segmento
 Alto Valor              18         208.89             38.0%      R$ 60,899,552.84
Médio Valor              17         202.82            34.85%      R$ 52,426,286.98
Baixo Valor              14         191.93            27.16%      R$ 44,488,620.07


In [ ]:
query = """
SELECT
    estado,
    COUNT(DISTINCT id_client) AS total_clientes,
    COUNT(*) AS total_compras,
    ROUND(SUM(total), 2) AS total_faturado

FROM base_completa
GROUP BY estado
ORDER BY total_clientes DESC
"""

df_clientes_estado = pd.read_sql(query, conn)
df_clientes_estado["total_faturado"] = df_clientes_estado["total_faturado"].apply(lambda x: f"R$ {x:,.2f}")
print("CLIENTES POR ESTADO")
print(df_clientes_estado.to_string(index=False))

CLIENTES POR ESTADO
estado  total_clientes  total_compras    total_faturado
    PA               8           1664 R$ 457,650,416.10
    BA               5            955 R$ 241,517,614.10
    TO               4            822 R$ 224,946,382.20
    SE               3            606 R$ 167,095,845.05
    PE               3            621 R$ 165,037,432.00
    PB               3            615 R$ 164,273,511.60
    MA               3            597 R$ 152,228,562.25
    CE               3            583 R$ 135,338,567.60
    AM               3            608 R$ 154,432,536.35
    RS               2            424 R$ 105,859,542.85
    PR               2            400 R$ 113,835,036.35
    MS               2            382  R$ 90,467,877.25
    AC               2            398 R$ 102,534,414.50
    SP               1            206  R$ 45,279,072.85
    SC               1            209  R$ 51,123,732.35
    RJ               1            190  R$ 64,003,343.75
    ES               1      

In [ ]:
query = """
SELECT
    full_name AS cliente,
    COUNT(*) AS total_compras,
    ROUND(SUM(total), 2) AS total_faturado,
    MIN(sale_date) AS primeira_compra,
    MAX(sale_date) AS ultima_compra

FROM base_completa
GROUP BY id_client, full_name
ORDER BY total_compras DESC
LIMIT 10
"""

df_frequencia = pd.read_sql(query, conn)
df_frequencia["total_faturado"] = df_frequencia["total_faturado"].apply(lambda x: f"R$ {x:,.2f}")
print("CLIENTES MAIS FREQUENTES")
print(df_frequencia.to_string(index=False))

CLIENTES MAIS FREQUENTES
                               cliente  total_compras   total_faturado primeira_compra ultima_compra
                     Márcia Figueiredo            222 R$ 72,187,369.50      2023-01-02    2024-12-29
              Lucas Guedes Cunha Lopes            218 R$ 66,788,855.35      2023-01-01    2024-12-30
  Fernanda Azevedo Soares Nunes Vieira            220 R$ 65,652,931.35      2023-01-07    2024-12-31
       Carla Lopes Alves Pacheco Rocha            233 R$ 64,171,776.55      2023-01-09    2024-12-29
               Gabriela Barros Lacerda            190 R$ 64,003,343.75      2023-01-04    2024-12-28
            Francisca Ribeiro Pinheiro            215 R$ 62,791,038.15      2023-01-09    2024-12-31
Femininos Antunes Lopes Ribeiro Amaral            217 R$ 62,028,628.95      2023-01-04    2024-12-30
                      Bianca Rodrigues            204 R$ 60,826,837.25      2023-01-01    2024-12-26
 Daniela Borges Vieira Farias Mendonça            198 R$ 59,581,39

In [ ]:
query = """
SELECT
    full_name AS cliente,
    SUM(CASE WHEN strftime('%Y', sale_date) = '2023'
        THEN total ELSE 0 END) AS faturado_2023,
    SUM(CASE WHEN strftime('%Y', sale_date) = '2024'
        THEN total ELSE 0 END) AS faturado_2024

FROM base_completa
GROUP BY id_client, full_name
HAVING faturado_2023 > 0 AND faturado_2024 > 0
ORDER BY faturado_2024 DESC
LIMIT 10
"""

df_recorrentes = pd.read_sql(query, conn)
df_recorrentes["faturado_2023"] = df_recorrentes["faturado_2023"].apply(lambda x: f"R$ {x:,.2f}")
df_recorrentes["faturado_2024"] = df_recorrentes["faturado_2024"].apply(lambda x: f"R$ {x:,.2f}")
print("CLIENTES RECORRENTES (2023 e 2024)")
print(df_recorrentes.to_string(index=False))

CLIENTES RECORRENTES (2023 e 2024)
                              cliente    faturado_2023    faturado_2024
                     Bianca Rodrigues R$ 20,680,709.25 R$ 40,146,128.00
Daniela Borges Vieira Farias Mendonça R$ 23,087,759.65 R$ 36,493,639.10
        Ana Silva Costa Farias Coelho R$ 22,939,998.05 R$ 36,186,836.30
 Fernanda Azevedo Soares Nunes Vieira R$ 29,848,028.75 R$ 35,804,902.60
                    Márcia Figueiredo R$ 37,361,787.70 R$ 34,825,581.80
             Lucas Guedes Cunha Lopes R$ 33,214,226.90 R$ 33,574,628.45
                       Mateus Antunes R$ 25,297,310.70 R$ 31,796,020.45
         Gabriela Silva Vieira Amaral R$ 24,573,925.20 R$ 29,965,984.40
             Carlos Guimarães Martins R$ 27,007,126.40 R$ 29,780,041.60
             Luiz Borges Gomes Araújo R$ 28,808,754.00 R$ 29,264,987.00


In [ ]:
conn.close()
print("Conexão fechada")

✅ Conexão fechada!
